# Agentic AI Workbench

This notebook demonstrates the project's tool-driven decision agent as a separate research track.

Focus:
- hazard lookup
- compliance checking
- disposal recommendation generation
- tool-trace transparency for each decision

The notebook does not alter the main pipeline. It runs the existing agent over curated scenarios and saves a reproducible casebook.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "agent").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from agent.agent import AgentInput, EwasteDecisionAgent

OUTPUT_DIR = PROJECT_ROOT / "models" / "agentic"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PROJECT_ROOT

d:\Github Desktop\ewaste_vit_project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


WindowsPath('D:/Github Desktop/ewaste_vit_project')

In [2]:
sample_cases = [
    {"component": "Battery", "confidence": 0.93},
    {"component": "PCB", "confidence": 0.81},
    {"component": "Printer", "confidence": 0.58},
    {"component": "Keyboard", "confidence": 0.76},
    {"component": "Refrigerator", "confidence": 0.67},
    {"component": "Mobile", "confidence": 0.42},
]

agent = EwasteDecisionAgent(confidence_threshold=0.70)
records = []
casebook = []

for case in sample_cases:
    decision = agent.run(AgentInput(component=case["component"], confidence=case["confidence"]))
    casebook.append({"input": case, "decision": decision})
    records.append(
        {
            "component": case["component"],
            "confidence": case["confidence"],
            "hazard_level": decision.get("hazard_level"),
            "requires_human_review": decision.get("requires_human_review"),
            "agent_mode": decision.get("agent_mode"),
            "llm_provider": decision.get("llm_provider"),
            "explanation_source": decision.get("explanation_source"),
            "short_recommendation": decision.get("short_recommendation"),
        }
    )

with (OUTPUT_DIR / "agentic_workbench_cases.json").open("w", encoding="utf-8") as fp:
    json.dump(casebook, fp, indent=2)

pd.DataFrame(records)

,component,confidence,hazard_level,requires_human_review,agent_mode,llm_provider,explanation_source,short_recommendation
0,Battery,0.93,HIGH,False,llm_augmented_tool_agent,groq,llm,send to hazardous battery recycling facility
1,PCB,0.81,HIGH,False,llm_augmented_tool_agent,groq,llm,send to certified ewaste recycler for metal re...
2,Printer,0.58,MEDIUM,True,llm_augmented_tool_agent,groq,llm,route to ewaste stream with toner-safe handling
3,Keyboard,0.76,LOW,False,llm_augmented_tool_agent,groq,llm,route to plastics and small-ewaste stream
4,Refrigerator,0.67,HIGH,True,llm_augmented_tool_agent,groq,llm,"recover refrigerant first, then dismantle in c..."
5,Mobile,0.42,HIGH,True,llm_augmented_tool_agent,groq,llm,send to certified electronics recycler


In [3]:
focus_case = casebook[2]
print("Input:", focus_case["input"])
print("\nTool trace:")
for step in focus_case["decision"].get("tool_trace", []):
    print(f"- {step['step']}: {step['summary']}")

focus_case["decision"]

Input: {'component': 'Printer', 'confidence': 0.58}

Tool trace:
- hazard_lookup: Printer mapped to MEDIUM risk with material profile plastic shell, toner residue, pcb.
- regulation_check: Confidence 58.00% evaluated against threshold 70.00%; human review = yes.
- disposal_recommendation: Recommended pathway: route to ewaste stream with toner-safe handling. Explanation source: llm.


{'component': 'Printer',
 'hazard_level': 'MEDIUM',
 'material_profile': 'plastic shell, toner residue, pcb',
 'disposal_pathway': 'route to ewaste stream with toner-safe handling',
 'sdg_target': 'SDG 12.4',
 'compliance_flag': True,
 'requires_human_review': True,
 'confidence_threshold': 0.7,
 'short_recommendation': 'route to ewaste stream with toner-safe handling',
 'explanation': 'The printer’s medium‑hazard rating stems mainly from heavy metals in the PCB, residual toner containing carbon black and potential additives, and the plastic shell’s potential for leaching phthalates. Route the unit to a certified e‑waste facility that offers toner‑safe segregation and PCB recycling; this ensures hazardous components are isolated, recovered, and disposed of per regulatory standards. Because confidence is only 58\u202f%, a manual review by a qualified e‑waste specialist is advised to confirm the pathway and verify compliance with SDG\u202f12.4.',
 'confidence': 0.58,
 'agent_mode': 'llm_